In [1]:
import os
import glob
import pandas as pd
import numpy as np

# 设置输入和输出文件夹路径
input_dir = r"F:\self_quant\data\data\合并后数据"
# 创建一个新的专门放 parquet 文件的文件夹
output_dir = r"F:\self_quant\data\data\合并后数据_带市值_parquet"

# 如果输出文件夹不存在，则自动创建
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 获取输入文件夹下所有的 csv 文件
file_pattern = os.path.join(input_dir, "*.csv")
file_list = glob.glob(file_pattern)

print(f"共找到 {len(file_list)} 个CSV文件，开始批量处理并转换为Parquet格式...")

success_count = 0

for file_path in file_list:
    file_name = os.path.basename(file_path)
    # 将输出的文件名后缀从 .csv 改为 .parquet
    parquet_file_name = file_name.replace('.csv', '.parquet')
    output_path = os.path.join(output_dir, parquet_file_name)

    try:
        # 读取原始 CSV 数据 (如果遇到编码报错，可以把 'gbk' 改成 'utf-8')
        df = pd.read_csv(file_path, encoding='gbk')

        # 检查是否包含计算所需的列
        if 'volume' in df.columns and 'turn' in df.columns and 'close' in df.columns:

            # 1. 避免除以0报错，将换手率为0的替换为空值(NaN)
            df['turn'] = df['turn'].replace(0, np.nan)

            # 2. 计算流通股本与流通市值（直接使用中文列名）
            # 换算逻辑：成交量 / (换手率 / 100)
            df['流通股本'] = df['volume'] / (df['turn'] / 100)
            df['流通市值'] = df['流通股本'] * df['close']

        else:
            print(f"⚠️ 文件 {file_name} 缺少必要的列 (volume, turn, close)，无法计算。")

        # 3. 将数据保存为 Parquet 格式，速度极快且压缩率高
        # 注意：parquet 不需要也不支持 encoding='gbk' 这种参数，它原生完美支持中文
        df.to_parquet(output_path, engine='pyarrow', index=False)
        success_count += 1

        # 每处理 500 个文件打印一次进度
        if success_count % 500 == 0:
            print(f"已处理 {success_count} 个文件...")

    except Exception as e:
        print(f"❌ 处理文件 {file_name} 时出错: {e}")

print(f"\n全部处理完成！成功处理了 {success_count} 个文件。")
print(f"带有【流通市值】和【流通股本】的Parquet数据已保存至：{output_dir}")

共找到 5443 个CSV文件，开始批量处理并转换为Parquet格式...
已处理 500 个文件...
已处理 1000 个文件...
已处理 1500 个文件...
已处理 2000 个文件...
已处理 2500 个文件...
已处理 3000 个文件...
已处理 3500 个文件...
已处理 4000 个文件...
已处理 4500 个文件...
已处理 5000 个文件...

全部处理完成！成功处理了 5443 个文件。
带有【流通市值】和【流通股本】的Parquet数据已保存至：F:\self_quant\data\data\合并后数据_带市值_parquet
